In [2]:
# -*- coding: utf-8 -*-
"""
CFPS 驱动的家庭财务模拟器（最终工程版）
------------------------------------------------
输出结构：
预测结果/
  1/
    财务档案_总表.xlsx
    财务档案_简表.xlsx
    month1/
      资产负债数据类别.xlsx
      收入支出数据类别.xlsx
    ...
    month6/
      资产负债数据类别.xlsx
      收入支出数据类别.xlsx
    汇总/
      资产负债表_6个月结构汇总.xlsx
      收入支出表_6个月结构汇总.xlsx
  2/
  3/
  ...

说明：
1. 财务档案只生成一次，作为静态基础画像
2. 月度数据在基础画像上，结合 CFPS 组内行为做 6 个月动态模拟
3. 资产负债表 / 收入支出表按模板结构原样输出
"""

import os
import random
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")


# =========================================================
# 0. 参数区
# =========================================================
BASE_DIR = r"D:\日均产出\产品设计思路 动态预测、支出收入动态管理、播客设计等 4.22"

# ---- CFPS Excel 路径 ----
CFPS_DIR = os.path.join(BASE_DIR, r"ABM算法\原始数据\Excel格式转换")

CFPS_FILES = {
    2018: {
        "econ": os.path.join(CFPS_DIR, "cfps2018famecon_202512.xlsx"),
        "conf": os.path.join(CFPS_DIR, "cfps2018famconf_202512.xlsx"),
    },
    2020: {
        "econ": os.path.join(CFPS_DIR, "cfps2020famecon_202306.xlsx"),
        "conf": os.path.join(CFPS_DIR, "cfps2020famconf_202306.xlsx"),
    },
    2022: {
        "econ": os.path.join(CFPS_DIR, "cfps2022famecon_202410.xlsx"),
        "conf": os.path.join(CFPS_DIR, "cfps2022famconf_202410.xlsx"),
    },
}

# ---- 模板表路径 ----
TEMPLATE_DIR = os.path.join(BASE_DIR, r"数据预测\原表")
ASSET_TEMPLATE = os.path.join(TEMPLATE_DIR, "资产负债数据类别.xlsx")
INCOME_TEMPLATE = os.path.join(TEMPLATE_DIR, "收入支出数据类别.xlsx")
PROFILE_TEMPLATE = os.path.join(TEMPLATE_DIR, "财务档案数据类别.xlsx")  # 用你最新的档案表

# ---- 输出路径 ----
OUTPUT_DIR = os.path.join(BASE_DIR, r"数据预测\预测结果")

# ---- 可调参数 ----
N_OUTPUT = 10
RANDOM_SEED = 42


# =========================================================
# 1. 基础工具函数
# =========================================================
def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def winsorize_series(s: pd.Series, lower=0.01, upper=0.99) -> pd.Series:
    s = safe_numeric(s.copy())
    valid = s.dropna()
    if len(valid) == 0:
        return s
    q1 = valid.quantile(lower)
    q2 = valid.quantile(upper)
    return s.clip(lower=q1, upper=q2)


def first_existing(df: pd.DataFrame, candidates: List[str]) -> pd.Series:
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        return pd.Series([np.nan] * len(df), index=df.index)

    out = pd.Series([np.nan] * len(df), index=df.index, dtype="float64")
    for c in existing:
        cur = safe_numeric(df[c]) if df[c].dtype != "O" else df[c]
        mask = out.isna() & cur.notna()
        out.loc[mask] = cur.loc[mask]
    return out


def read_excel_selected(path: str, wanted_cols: List[str]) -> pd.DataFrame:
    head = pd.read_excel(path, nrows=0)
    cols = [c for c in wanted_cols if c in head.columns]
    return pd.read_excel(path, usecols=cols)


def positive(x, default=0.0):
    if pd.isna(x):
        return default
    return max(float(x), 0.0)


def clip_int(x, low, high):
    return int(np.clip(int(round(x)), low, high))


def find_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = str(c).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def get_sheet_name(path: str, preferred: str = None) -> str:
    xls = pd.ExcelFile(path)
    sheets = xls.sheet_names
    if preferred and preferred in sheets:
        return preferred
    return sheets[0]


def robust_income_group(series: pd.Series) -> pd.Series:
    s = safe_numeric(series).copy()
    result = pd.Series(["未知"] * len(s), index=s.index, dtype="object")

    valid = s.dropna()
    if len(valid) == 0:
        return result

    nunique = valid.nunique()
    if nunique <= 1:
        result.loc[valid.index] = "单一收入层"
        return result

    q = min(4, nunique)
    try:
        cat = pd.qcut(valid, q=q, duplicates="drop")
        result.loc[valid.index] = cat.astype(str)
        return result
    except Exception:
        try:
            ranks = valid.rank(method="average")
            cat = pd.qcut(ranks, q=q, duplicates="drop")
            result.loc[valid.index] = cat.astype(str)
            return result
        except Exception:
            median_val = valid.median()
            result.loc[valid.index] = np.where(valid <= median_val, "低收入", "高收入")
            return result


def normalize_group_key(x) -> str:
    return str(x) if pd.notna(x) else "未知"


# =========================================================
# 2. 读取 CFPS：构建家庭样本池
# =========================================================
def load_cfps_wave(year: int, econ_path: str, conf_path: str) -> pd.DataFrame:
    fid_col = f"fid{str(year)[-2:]}"
    urban_col = f"fid_urban{str(year)[-2:]}"
    size_col = f"familysize{str(year)[-2:]}"
    birth_col = "tb1y_a_p"

    econ_candidates = [
        fid_col,
        "finc", "finc_est", "fincome1",
        "expense", "fincexp",
        "savings", "financial_product", "finance_asset",
        "houseasset_gross", "houseasset_net",
        "house_debts", "nonhousing_debts",
        "durables_asset", "fixed_asset", "land_asset",
        "otherhousevalue", "total_asset"
    ]

    conf_candidates = [
        fid_col,
        urban_col,
        size_col,
        birth_col,
        f"tb3_a{str(year)[-2:]}_p",
        f"hukou_a{str(year)[-2:]}_p",
    ]

    econ = read_excel_selected(econ_path, econ_candidates)
    conf = read_excel_selected(conf_path, conf_candidates)

    if fid_col not in econ.columns:
        raise ValueError(f"{econ_path} 中找不到主键 {fid_col}")
    if fid_col not in conf.columns:
        raise ValueError(f"{conf_path} 中找不到主键 {fid_col}")

    df = pd.merge(econ, conf, on=fid_col, how="inner")

    df["family_id"] = df[fid_col]
    df["year"] = year

    df["income"] = first_existing(df, ["finc_est", "finc", "fincome1"])
    df["expense_total"] = first_existing(df, ["expense", "fincexp"])

    df["savings_amt"] = first_existing(df, ["savings"])
    df["fin_product_amt"] = first_existing(df, ["financial_product"])
    df["finance_asset_amt"] = first_existing(df, ["finance_asset"])
    df["house_asset_amt"] = first_existing(df, ["houseasset_net", "houseasset_gross"])
    df["durables_asset_amt"] = first_existing(df, ["durables_asset"])
    df["fixed_asset_amt"] = first_existing(df, ["fixed_asset"])
    df["land_asset_amt"] = first_existing(df, ["land_asset"])
    df["other_house_amt"] = first_existing(df, ["otherhousevalue"])
    df["total_asset_amt"] = first_existing(df, ["total_asset"])

    df["house_debt_amt"] = first_existing(df, ["house_debts"])
    df["other_debt_amt"] = first_existing(df, ["nonhousing_debts"])

    df["urban"] = df[urban_col] if urban_col in df.columns else np.nan
    df["family_size"] = safe_numeric(df[size_col]) if size_col in df.columns else np.nan
    df["birth_year"] = safe_numeric(df[birth_col]) if birth_col in df.columns else np.nan
    df["age"] = year - df["birth_year"]

    num_cols = [
        "income", "expense_total",
        "savings_amt", "fin_product_amt", "finance_asset_amt",
        "house_asset_amt", "durables_asset_amt", "fixed_asset_amt",
        "land_asset_amt", "other_house_amt", "total_asset_amt",
        "house_debt_amt", "other_debt_amt",
        "family_size", "age"
    ]
    for c in num_cols:
        df[c] = safe_numeric(df[c])

    for c in [
        "income", "expense_total", "savings_amt", "fin_product_amt", "finance_asset_amt",
        "house_asset_amt", "durables_asset_amt", "fixed_asset_amt",
        "land_asset_amt", "other_house_amt", "total_asset_amt",
        "house_debt_amt", "other_debt_amt"
    ]:
        df.loc[df[c] < 0, c] = np.nan

    df.loc[(df["age"] < 20) | (df["age"] > 85), "age"] = np.nan
    df.loc[(df["family_size"] < 1) | (df["family_size"] > 12), "family_size"] = np.nan

    df["liquid_asset"] = df["savings_amt"].fillna(0)
    df["invest_asset"] = df["fin_product_amt"].fillna(0) + df["finance_asset_amt"].fillna(0)
    df["selfuse_asset"] = (
        df["house_asset_amt"].fillna(0)
        + 0.60 * df["durables_asset_amt"].fillna(0)
        + 0.40 * df["fixed_asset_amt"].fillna(0)
    )
    df["other_asset"] = (
        df["total_asset_amt"].fillna(0)
        - df["liquid_asset"].fillna(0)
        - df["invest_asset"].fillna(0)
        - df["selfuse_asset"].fillna(0)
    ).clip(lower=0)

    df["total_debt"] = df["house_debt_amt"].fillna(0) + df["other_debt_amt"].fillna(0)
    df["total_asset_amt"] = df["total_asset_amt"].fillna(
        df["liquid_asset"] + df["invest_asset"] + df["selfuse_asset"] + df["other_asset"]
    )
    df["net_asset"] = df["total_asset_amt"] - df["total_debt"]

    df["expense_ratio"] = df["expense_total"] / df["income"].replace(0, np.nan)
    df["debt_ratio"] = df["total_debt"] / df["total_asset_amt"].replace(0, np.nan)
    df["invest_ratio"] = df["invest_asset"] / df["total_asset_amt"].replace(0, np.nan)
    df["liquid_ratio"] = df["liquid_asset"] / df["total_asset_amt"].replace(0, np.nan)
    df["house_ratio"] = df["house_asset_amt"] / df["total_asset_amt"].replace(0, np.nan)
    df["has_house"] = (df["house_asset_amt"].fillna(0) > 0).astype(int)

    keep_cols = [
        "family_id", "year", "urban", "family_size", "age",
        "income", "expense_total",
        "liquid_asset", "invest_asset", "selfuse_asset", "other_asset",
        "house_asset_amt", "durables_asset_amt", "fixed_asset_amt",
        "total_asset_amt", "house_debt_amt", "other_debt_amt", "total_debt", "net_asset",
        "expense_ratio", "debt_ratio", "invest_ratio", "liquid_ratio", "house_ratio", "has_house"
    ]
    return df[keep_cols]


def build_cfps_pool() -> Tuple[pd.DataFrame, pd.DataFrame]:
    waves = []
    for year, fp in CFPS_FILES.items():
        print(f"正在读取 {year} 年 CFPS ...")
        wave = load_cfps_wave(year, fp["econ"], fp["conf"])
        waves.append(wave)

    pool = pd.concat(waves, ignore_index=True)

    pool = pool[
        pool["income"].notna() &
        pool["expense_total"].notna() &
        pool["income"].gt(0)
    ].copy()

    pool["family_size"] = pool["family_size"].fillna(pool["family_size"].median())
    pool["age"] = pool["age"].fillna(pool["age"].median())
    pool["urban"] = pool["urban"].fillna("缺失")

    for c in [
        "income", "expense_total", "liquid_asset", "invest_asset", "selfuse_asset", "other_asset",
        "house_asset_amt", "durables_asset_amt", "fixed_asset_amt",
        "total_asset_amt", "house_debt_amt", "other_debt_amt", "total_debt", "net_asset"
    ]:
        pool[c] = winsorize_series(pool[c], 0.01, 0.99)

    pool["income_group"] = robust_income_group(pool["income"])
    pool["debt_group"] = pd.cut(
        pool["debt_ratio"].fillna(0),
        bins=[-0.01, 0.10, 0.30, 0.60, 1.50, 999],
        labels=["低杠杆", "温和杠杆", "中杠杆", "高杠杆", "超高杠杆"]
    ).astype(str)

    pool["family_type"] = (
        pool["income_group"].astype(str) + "_" +
        pool["debt_group"].astype(str) + "_" +
        pool["has_house"].astype(str)
    )

    # 组内行为参数，用于月度演化
    pool["income_growth_proxy"] = pool.groupby("family_type")["income"].pct_change().replace([np.inf, -np.inf], np.nan)
    pool["expense_growth_proxy"] = pool.groupby("family_type")["expense_total"].pct_change().replace([np.inf, -np.inf], np.nan)
    pool["asset_growth_proxy"] = pool.groupby("family_type")["total_asset_amt"].pct_change().replace([np.inf, -np.inf], np.nan)
    pool["debt_growth_proxy"] = pool.groupby("family_type")["total_debt"].pct_change().replace([np.inf, -np.inf], np.nan)

    grp_stats = pool.groupby("family_type").agg(
        income_mean=("income", "mean"),
        income_std=("income", "std"),
        expense_ratio_mean=("expense_ratio", "mean"),
        expense_ratio_std=("expense_ratio", "std"),
        liquid_ratio_mean=("liquid_ratio", "mean"),
        invest_ratio_mean=("invest_ratio", "mean"),
        house_ratio_mean=("house_ratio", "mean"),
        debt_ratio_mean=("debt_ratio", "mean"),
        has_house_prob=("has_house", "mean"),
        inc_g_mean=("income_growth_proxy", "mean"),
        inc_g_std=("income_growth_proxy", "std"),
        exp_g_mean=("expense_growth_proxy", "mean"),
        exp_g_std=("expense_growth_proxy", "std"),
        ast_g_mean=("asset_growth_proxy", "mean"),
        ast_g_std=("asset_growth_proxy", "std"),
        debt_g_mean=("debt_growth_proxy", "mean"),
        debt_g_std=("debt_growth_proxy", "std"),
        n=("family_id", "count")
    ).reset_index()

    grp_stats = grp_stats.fillna(0)
    return pool, grp_stats


# =========================================================
# 3. 抽样一个基础家庭（静态）
# =========================================================
def random_from_group(pool: pd.DataFrame):
    donor = pool.sample(1, random_state=np.random.randint(0, 10**9)).iloc[0]
    group = donor["family_type"]
    sub = pool[pool["family_type"] == group]
    if len(sub) < 30:
        sub = pool
    return donor, sub


def draw_ratio(series: pd.Series, default: float, low: float, high: float) -> float:
    s = safe_numeric(series).dropna()
    if len(s) == 0:
        val = default
    else:
        val = float(s.sample(1).iloc[0])
    return float(np.clip(val, low, high))


def generate_retirement_age(age: int, sex: str) -> int:
    if sex == "女":
        if age < 30:
            return int(np.random.choice([55, 58, 60], p=[0.15, 0.15, 0.70]))
        elif age < 45:
            return int(np.random.choice([55, 58, 60], p=[0.20, 0.20, 0.60]))
        else:
            return int(np.random.choice([55, 58, 60], p=[0.30, 0.25, 0.45]))
    else:
        if age < 30:
            return int(np.random.choice([60, 62, 65], p=[0.20, 0.20, 0.60]))
        elif age < 45:
            return int(np.random.choice([60, 62, 65], p=[0.30, 0.25, 0.45]))
        else:
            return int(np.random.choice([60, 62, 65], p=[0.40, 0.20, 0.40]))


def simulate_base_household(pool: pd.DataFrame) -> Dict:
    """
    生成基础用户画像（静态）
    """
    donor, grp = random_from_group(pool)

    age = clip_int(donor["age"] + np.random.normal(0, 3), 22, 75)
    family_size = clip_int(donor["family_size"] + np.random.choice([-1, 0, 0, 1]), 1, 8)
    urban = donor["urban"] if pd.notna(donor["urban"]) else "城镇"
    sex = np.random.choice(["男", "女"], p=[0.55, 0.45])

    if age < 28:
        child_num = np.random.choice([0, 1], p=[0.80, 0.20])
    elif age < 38:
        child_num = np.random.choice([0, 1, 2], p=[0.25, 0.50, 0.25])
    elif age < 50:
        child_num = np.random.choice([0, 1, 2, 3], p=[0.15, 0.45, 0.30, 0.10])
    else:
        child_num = np.random.choice([0, 1, 2], p=[0.25, 0.45, 0.30])

    family_type = donor["family_type"]

    # 基础收入支出
    income_annual = positive(donor["income"], 120000) * np.random.lognormal(mean=0, sigma=0.12)
    expense_ratio = draw_ratio(grp["expense_ratio"], default=0.65, low=0.25, high=0.92)
    expense_annual = np.clip(income_annual * np.random.normal(expense_ratio, 0.05), income_annual * 0.25, income_annual * 0.96)

    # 基础资产
    total_asset = positive(donor["total_asset_amt"], income_annual * np.random.uniform(1.5, 8))
    total_asset = max(total_asset * np.random.lognormal(0, 0.12), income_annual * np.random.uniform(0.8, 5.5))

    # 是否有房
    has_house = bool(donor["has_house"]) if pd.notna(donor["has_house"]) else (np.random.rand() < 0.6)

    # 结构性负债：修复“负债为0”
    if income_annual < 150000:
        debt_ratio = np.random.uniform(0.20, 0.60)
    elif income_annual < 300000:
        debt_ratio = np.random.uniform(0.10, 0.40)
    else:
        debt_ratio = np.random.uniform(0.05, 0.30)

    if has_house:
        debt_ratio += np.random.uniform(0.05, 0.18)

    debt_ratio = float(np.clip(debt_ratio, 0.03, 0.75))
    total_debt = total_asset * debt_ratio

    # 资产分配
    liquid_ratio = draw_ratio(grp["liquid_ratio"], default=0.18, low=0.05, high=0.55)
    invest_ratio = draw_ratio(grp["invest_ratio"], default=0.15, low=0.02, high=0.45)
    house_ratio = draw_ratio(grp["house_ratio"], default=0.45 if has_house else 0.0, low=0.10 if has_house else 0.0, high=0.85 if has_house else 0.0)

    liquid_asset = total_asset * liquid_ratio
    invest_asset = total_asset * invest_ratio
    house_asset = total_asset * house_ratio if has_house else 0.0
    selfuse_other = max(total_asset - liquid_asset - invest_asset - house_asset, 0)

    # 房贷与其他负债
    if has_house and house_asset > 0:
        house_debt = total_debt * np.random.uniform(0.50, 0.88)
    else:
        house_debt = 0.0
    other_debt = max(total_debt - house_debt, 0.0)

    # 流动资产拆分
    cash = liquid_asset * np.random.uniform(0.30, 0.60)
    money_fund = liquid_asset * np.random.uniform(0.15, 0.35)
    other_liq_deposit = max(liquid_asset - cash - money_fund, 0)

    # 投资资产拆分
    risk_score = 0.5
    if age < 40:
        risk_score += 0.1
    if invest_ratio > 0.20:
        risk_score += 0.1
    if house_asset > total_asset * 0.60:
        risk_score -= 0.05
    risk_score = np.clip(risk_score, 0.2, 0.8)

    stock_share = np.random.uniform(0.08, 0.25) + (risk_score - 0.5) * 0.18
    fund_share = np.random.uniform(0.15, 0.30) + (risk_score - 0.5) * 0.08
    bond_share = np.random.uniform(0.05, 0.15)
    fixed_share = np.random.uniform(0.15, 0.30)
    fx_share = np.random.uniform(0.00, 0.05)
    inv_house_share = np.random.uniform(0.00, 0.12) if has_house else np.random.uniform(0.00, 0.05)

    shares = np.array([fixed_share, fx_share, stock_share, bond_share, fund_share, inv_house_share], dtype=float)
    shares = np.clip(shares, 0.0001, None)
    shares = shares / shares.sum()

    fixed_deposit = invest_asset * shares[0]
    fx_deposit = invest_asset * shares[1]
    stock = invest_asset * shares[2]
    bond = invest_asset * shares[3]
    fund = invest_asset * shares[4]
    inv_house = invest_asset * shares[5]
    other_inv = max(invest_asset - (fixed_deposit + fx_deposit + stock + bond + fund + inv_house), 0)

    # 自用资产拆分
    car_asset = selfuse_other * np.random.uniform(0.20, 0.50)
    other_selfuse = max(selfuse_other - car_asset, 0)

    # 负债拆分
    credit_card_debt = other_debt * np.random.uniform(0.10, 0.25)
    small_consume_loan = other_debt * np.random.uniform(0.15, 0.35)
    other_liq_debt = other_debt * np.random.uniform(0.05, 0.20)

    invest_loan_total = 0.0
    if invest_asset > income_annual * 0.5 and np.random.rand() < 0.25:
        invest_loan_total = other_debt * np.random.uniform(0.08, 0.25)

    financial_invest_loan = invest_loan_total * np.random.uniform(0.30, 0.60)
    industry_invest_loan = invest_loan_total * np.random.uniform(0.10, 0.30)
    inv_house_loan = invest_loan_total * np.random.uniform(0.05, 0.20)
    other_invest_debt = max(invest_loan_total - financial_invest_loan - industry_invest_loan - inv_house_loan, 0)

    car_loan = 0.0
    if car_asset > 50000 and np.random.rand() < 0.45:
        car_loan = car_asset * np.random.uniform(0.15, 0.40)

    other_selfuse_debt = max(other_debt - credit_card_debt - small_consume_loan - other_liq_debt - invest_loan_total - car_loan, 0)

    # 收入拆分（年度）
    passive_income_ratio = np.clip((invest_asset / max(total_asset, 1)) * 0.18 + np.random.uniform(0.02, 0.06), 0.02, 0.18)
    transfer_income_ratio = np.clip(np.random.uniform(0.01, 0.06), 0.00, 0.08)
    other_income_ratio = np.clip(np.random.uniform(0.00, 0.04), 0.00, 0.05)

    passive_income = income_annual * passive_income_ratio
    transfer_income = income_annual * transfer_income_ratio
    other_income = income_annual * other_income_ratio
    active_income = max(income_annual - passive_income - transfer_income - other_income, 0)

    wage = active_income * np.random.uniform(0.60, 0.85)
    labor = active_income * np.random.uniform(0.05, 0.20)
    business = active_income * np.random.uniform(0.00, 0.15)
    pension = active_income * np.random.uniform(0.00, 0.12) if age >= 55 else 0.0
    other_active = max(active_income - wage - labor - business - pension, 0)

    gift = transfer_income * np.random.uniform(0.10, 0.30)
    subsidy = transfer_income * np.random.uniform(0.20, 0.45)
    inheritance = transfer_income * np.random.uniform(0.00, 0.15)
    other_transfer = max(transfer_income - gift - subsidy - inheritance, 0)

    refund = other_income * np.random.uniform(0.05, 0.20)
    windfall = other_income * np.random.uniform(0.10, 0.30)
    other_income_sub = max(other_income - refund - windfall, 0)

    rent_income = passive_income * np.random.uniform(0.05, 0.20) if house_asset > 0 else 0.0
    interest_income = passive_income * np.random.uniform(0.10, 0.25)
    royalty_income = passive_income * np.random.uniform(0.00, 0.05)
    financial_realization = passive_income * np.random.uniform(0.15, 0.35)
    physical_realization = passive_income * np.random.uniform(0.05, 0.15)
    secondhand_sale = passive_income * np.random.uniform(0.00, 0.05)
    other_passive = max(
        passive_income - rent_income - interest_income - royalty_income
        - financial_realization - physical_realization - secondhand_sale, 0
    )

    # 支出拆分（年度）
    finance_cost = (
        house_debt * np.random.uniform(0.012, 0.035) +
        other_debt * np.random.uniform(0.01, 0.04)
    )
    finance_cost = min(finance_cost, expense_annual * 0.28)
    living_expense = max(expense_annual - finance_cost, 0)

    food = living_expense * np.random.uniform(0.14, 0.24)
    housing_cost = living_expense * np.random.uniform(0.10, 0.18)
    commute = living_expense * np.random.uniform(0.05, 0.12)
    clothing = living_expense * np.random.uniform(0.03, 0.08)
    daily_goods = living_expense * np.random.uniform(0.04, 0.10)
    medical = living_expense * np.random.uniform(0.02, 0.08) + (6000 if age >= 55 else 0)
    other_life = living_expense * np.random.uniform(0.03, 0.10)

    learning = living_expense * np.random.uniform(0.01, 0.05)
    health_inv = living_expense * np.random.uniform(0.01, 0.05)
    hobby = living_expense * np.random.uniform(0.01, 0.04)
    other_growth = living_expense * np.random.uniform(0.00, 0.03)

    support_old = living_expense * np.random.uniform(0.02, 0.10) if age >= 30 else 0
    child_support = living_expense * np.random.uniform(0.04, 0.18) if child_num > 0 else 0
    pet = living_expense * np.random.uniform(0.00, 0.03)
    other_family = living_expense * np.random.uniform(0.00, 0.03)

    gifts_social = living_expense * np.random.uniform(0.01, 0.05)
    business_social = living_expense * np.random.uniform(0.00, 0.05)
    other_social = living_expense * np.random.uniform(0.00, 0.03)

    tax_penalty = living_expense * np.random.uniform(0.00, 0.03)
    accident_loss = living_expense * np.random.uniform(0.00, 0.03)

    used = (
        food + housing_cost + commute + clothing + daily_goods + medical + other_life +
        learning + health_inv + hobby + other_growth +
        support_old + child_support + pet + other_family +
        gifts_social + business_social + other_social +
        tax_penalty + accident_loss
    )
    other_uncat = max(living_expense - used, 0)

    premium = finance_cost * np.random.uniform(0.20, 0.45)
    interest_cost = finance_cost * np.random.uniform(0.35, 0.65)
    other_finance_cost = max(finance_cost - premium - interest_cost, 0)

    total_assets_final = (
        cash + money_fund + other_liq_deposit +
        fixed_deposit + fx_deposit + stock + bond + fund + inv_house + other_inv +
        house_asset + car_asset + other_selfuse
    )
    total_debts_final = (
        credit_card_debt + small_consume_loan + other_liq_debt +
        financial_invest_loan + industry_invest_loan + inv_house_loan + other_invest_debt +
        house_debt + car_loan + other_selfuse_debt
    )
    net_asset_final = total_assets_final - total_debts_final

    retirement_age = generate_retirement_age(age, sex)
    target_house_year = np.random.choice([2027, 2028, 2029, 2030, 2031]) if not has_house else ""
    target_house_area = clip_int(np.random.normal(95, 25), 50, 180) if not has_house else ""
    child_edu_stage = ""
    child_edu_fund = ""
    if child_num > 0:
        child_edu_stage = np.random.choice(["小学", "初中", "高中", "本科"])
        child_edu_fund = clip_int(np.random.normal(30, 12), 5, 80)

    return {
        "family_type": family_type,
        "age": age,
        "family_size": family_size,
        "urban": urban,
        "sex": sex,
        "child_num": int(child_num),

        # 资产
        "现金": cash,
        "货币基金/活期": money_fund,
        "其他流动性存款": other_liq_deposit,
        "定期存款": fixed_deposit,
        "外币存款": fx_deposit,
        "股票投资": stock,
        "债券投资": bond,
        "基金投资": fund,
        "投资性房地产": inv_house,
        "其他投资性资产": other_inv,
        "自用房产": house_asset,
        "自用汽车": car_asset,
        "其他自用性资产": other_selfuse,
        "其他资产": 0.0,

        # 负债
        "信用卡负债": credit_card_debt,
        "小额消费信贷": small_consume_loan,
        "其他流动性负债": other_liq_debt,
        "金融投资借款": financial_invest_loan,
        "实业投资借款": industry_invest_loan,
        "投资性房地产贷款": inv_house_loan,
        "其他投资性负债": other_invest_debt,
        "自住房按揭贷款": house_debt,
        "自用车按揭贷款": car_loan,
        "其他自用性负债": other_selfuse_debt,
        "其他负债": 0.0,
        "净值": net_asset_final,

        # 收入
        "工资薪金": wage,
        "劳务报酬": labor,
        "经营所得": business,
        "养老金": pension,
        "其他主动收入": other_active,
        "礼物礼金": gift,
        "补贴补助": subsidy,
        "继承收入": inheritance,
        "其他转移收入": other_transfer,
        "退税收入": refund,
        "偶然所得": windfall,
        "其他所得": other_income_sub,

        # 支出
        "饮食开支": food,
        "居住成本": housing_cost,
        "通勤交通": commute,
        "衣着形象": clothing,
        "日常用品": daily_goods,
        "医疗支出": medical,
        "其他生活消费": other_life,

        "学习提升": learning,
        "健康投资": health_inv,
        "兴趣培养": hobby,
        "其他个人成长消费": other_growth,

        "赡养老人": support_old,
        "子女抚养": child_support,
        "宠物养护": pet,
        "其他家庭成员消费": other_family,

        "人情往来": gifts_social,
        "商务应酬": business_social,
        "其他人际关系消费": other_social,

        "税费罚金": tax_penalty,
        "意外损失": accident_loss,
        "其他无法归类支出": other_uncat,

        # 被动收入 / 财务支出
        "理赔收入": 0.0,
        "租金收入": rent_income,
        "利息收入": interest_income,
        "版权收入": royalty_income,
        "金融资产变现损益": financial_realization,
        "实物资产变现损益": physical_realization,
        "二手物品出售": secondhand_sale,
        "其他被动收入": other_passive,

        "保费支出": premium,
        "利息费用": interest_cost,
        "其他财务支出": other_finance_cost,

        # 档案信息
        "退休年龄": retirement_age,
        "购房年份": target_house_year,
        "购房面积": target_house_area,
        "教育阶段": child_edu_stage,
        "教育资金储备": child_edu_fund,
    }


# =========================================================
# 4. 月度演化（组内行为驱动）
# =========================================================
ASSET_ITEM_KEYS = [
    "现金", "货币基金/活期", "其他流动性存款",
    "定期存款", "外币存款", "股票投资", "债券投资", "基金投资",
    "投资性房地产", "其他投资性资产",
    "自用房产", "自用汽车", "其他自用性资产", "其他资产"
]

DEBT_ITEM_KEYS = [
    "信用卡负债", "小额消费信贷", "其他流动性负债",
    "金融投资借款", "实业投资借款", "投资性房地产贷款", "其他投资性负债",
    "自住房按揭贷款", "自用车按揭贷款", "其他自用性负债", "其他负债"
]

INCOME_ITEM_KEYS = [
    "工资薪金", "劳务报酬", "经营所得", "养老金", "其他主动收入",
    "礼物礼金", "补贴补助", "继承收入", "其他转移收入",
    "退税收入", "偶然所得", "其他所得",
    "理赔收入", "租金收入", "利息收入", "版权收入",
    "金融资产变现损益", "实物资产变现损益", "二手物品出售", "其他被动收入"
]

EXPENSE_ITEM_KEYS = [
    "饮食开支", "居住成本", "通勤交通", "衣着形象", "日常用品", "医疗支出", "其他生活消费",
    "学习提升", "健康投资", "兴趣培养", "其他个人成长消费",
    "赡养老人", "子女抚养", "宠物养护", "其他家庭成员消费",
    "人情往来", "商务应酬", "其他人际关系消费",
    "税费罚金", "意外损失", "其他无法归类支出",
    "保费支出", "利息费用", "其他财务支出"
]


def get_group_behavior(grp_stats: pd.DataFrame, family_type: str) -> Dict:
    row = grp_stats[grp_stats["family_type"] == family_type]
    if row.empty:
        return {
            "inc_g_mean": 0.002, "inc_g_std": 0.01,
            "exp_g_mean": 0.003, "exp_g_std": 0.012,
            "ast_g_mean": 0.002, "ast_g_std": 0.01,
            "debt_g_mean": -0.005, "debt_g_std": 0.01,
        }
    row = row.iloc[0]
    return {
        "inc_g_mean": row["inc_g_mean"] if pd.notna(row["inc_g_mean"]) else 0.002,
        "inc_g_std": max(row["inc_g_std"], 0.008) if pd.notna(row["inc_g_std"]) else 0.01,
        "exp_g_mean": row["exp_g_mean"] if pd.notna(row["exp_g_mean"]) else 0.003,
        "exp_g_std": max(row["exp_g_std"], 0.008) if pd.notna(row["exp_g_std"]) else 0.012,
        "ast_g_mean": row["ast_g_mean"] if pd.notna(row["ast_g_mean"]) else 0.002,
        "ast_g_std": max(row["ast_g_std"], 0.008) if pd.notna(row["ast_g_std"]) else 0.01,
        "debt_g_mean": row["debt_g_mean"] if pd.notna(row["debt_g_mean"]) else -0.005,
        "debt_g_std": max(row["debt_g_std"], 0.008) if pd.notna(row["debt_g_std"]) else 0.01,
    }


def evolve_month_state(base_state: Dict, behavior: Dict, month_idx: int) -> Dict:
    """
    在基础静态画像上，按组内行为生成 month1~month6。
    财务档案不变；收入支出和资产负债子项随月变化。
    """
    state = dict(base_state)

    inc_growth = np.random.normal(behavior["inc_g_mean"], behavior["inc_g_std"])
    exp_growth = np.random.normal(behavior["exp_g_mean"], behavior["exp_g_std"])
    ast_growth = np.random.normal(behavior["ast_g_mean"], behavior["ast_g_std"])
    debt_growth = np.random.normal(behavior["debt_g_mean"], behavior["debt_g_std"])

    # 稳一点，别波动过大
    inc_growth = float(np.clip(inc_growth, -0.10, 0.12))
    exp_growth = float(np.clip(exp_growth, -0.10, 0.15))
    ast_growth = float(np.clip(ast_growth, -0.10, 0.12))
    debt_growth = float(np.clip(debt_growth, -0.12, 0.08))

    # 每个月在基础上叠加
    income_factor = (1 + inc_growth) ** month_idx
    expense_factor = (1 + exp_growth) ** month_idx
    asset_factor = (1 + ast_growth) ** month_idx
    debt_factor = max((1 + debt_growth) ** month_idx, 0.30)

    # 收入项演化
    for k in INCOME_ITEM_KEYS:
        if k in state:
            # 工资薪金稍稳，偶然所得波动更大
            if k == "工资薪金":
                item_factor = income_factor * np.random.uniform(0.97, 1.03)
            elif k in ["偶然所得", "其他所得", "金融资产变现损益", "实物资产变现损益", "二手物品出售"]:
                item_factor = income_factor * np.random.uniform(0.70, 1.35)
            else:
                item_factor = income_factor * np.random.uniform(0.90, 1.10)
            state[k] = max(state[k] / 12.0 * item_factor, 0)

    # 支出项演化
    for k in EXPENSE_ITEM_KEYS:
        if k in state:
            if k in ["医疗支出", "意外损失"]:
                item_factor = expense_factor * np.random.uniform(0.80, 1.40)
            elif k in ["子女抚养", "赡养老人", "居住成本"]:
                item_factor = expense_factor * np.random.uniform(0.95, 1.10)
            else:
                item_factor = expense_factor * np.random.uniform(0.90, 1.15)
            state[k] = max(state[k] / 12.0 * item_factor, 0)

    # 资产项演化
    for k in ASSET_ITEM_KEYS:
        if k in state:
            if k in ["股票投资", "基金投资"]:
                item_factor = asset_factor * np.random.uniform(0.90, 1.12)
            elif k in ["现金", "货币基金/活期", "其他流动性存款"]:
                item_factor = asset_factor * np.random.uniform(0.96, 1.05)
            elif k in ["自用房产", "自用汽车", "其他自用性资产"]:
                item_factor = asset_factor * np.random.uniform(0.99, 1.03)
            else:
                item_factor = asset_factor * np.random.uniform(0.94, 1.08)
            state[k] = max(state[k] * item_factor, 0)

    # 负债项演化
    for k in DEBT_ITEM_KEYS:
        if k in state:
            if k in ["自住房按揭贷款", "自用车按揭贷款"]:
                item_factor = debt_factor * np.random.uniform(0.97, 1.00)  # 更偏向缓慢下降
            elif k in ["信用卡负债", "小额消费信贷", "其他流动性负债"]:
                item_factor = debt_factor * np.random.uniform(0.92, 1.05)
            else:
                item_factor = debt_factor * np.random.uniform(0.95, 1.03)
            state[k] = max(state[k] * item_factor, 0)

    # 重新计算净值
    total_assets = sum(state.get(k, 0) for k in ASSET_ITEM_KEYS)
    total_debts = sum(state.get(k, 0) for k in DEBT_ITEM_KEYS)
    state["净值"] = total_assets - total_debts

    return state


# =========================================================
# 5. 填模板：按原表结构输出
# =========================================================
def fill_asset_template(template_path: str, sim: Dict) -> pd.DataFrame:
    sheet = get_sheet_name(template_path, preferred="报表")
    df = pd.read_excel(template_path, sheet_name=sheet).copy()

    secondary_col = find_column(df, ["二级分类"])
    if secondary_col is None:
        raise ValueError("资产负债模板中找不到‘二级分类’列")

    out_col = "预测值"
    if out_col not in df.columns:
        df[out_col] = np.nan

    for i in df.index:
        sub = str(df.loc[i, secondary_col]) if pd.notna(df.loc[i, secondary_col]) else ""
        if sub in sim:
            df.loc[i, out_col] = round(float(sim[sub]), 2)

    # 如果模板没有净值行，自动补
    all_text = df.astype(str).fillna("")
    has_net = all_text.apply(lambda x: x.str.contains("净值")).any().any()
    if not has_net:
        row = {c: np.nan for c in df.columns}
        first_col = df.columns[0]
        row[first_col] = ""
        if "AFP家财分类" in df.columns:
            row["AFP家财分类"] = "净值"
        if secondary_col in df.columns:
            row[secondary_col] = "净值"
        row[out_col] = round(float(sim["净值"]), 2)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

    return df


def fill_income_template(template_path: str, sim: Dict) -> pd.DataFrame:
    sheet = get_sheet_name(template_path, preferred="报表")
    df = pd.read_excel(template_path, sheet_name=sheet).copy()

    secondary_col = find_column(df, ["二级分类"])
    if secondary_col is None:
        raise ValueError("收入支出模板中找不到‘二级分类’列")

    out_col = "预测值"
    if out_col not in df.columns:
        df[out_col] = np.nan

    for i in df.index:
        sub = str(df.loc[i, secondary_col]) if pd.notna(df.loc[i, secondary_col]) else ""
        if sub in sim:
            df.loc[i, out_col] = round(float(sim[sub]), 2)

    return df


# =========================================================
# 6. 财务档案：总表 + 简表
# =========================================================
def _mark_profile_option(df: pd.DataFrame, cols: List[str], selected_value: str):
    for col in cols:
        if col in df.columns:
            mask = df[col].astype(str) == str(selected_value)
            df.loc[mask, "是否选中"] = "是"


def fill_profile_total(template_path: str, sim: Dict) -> pd.DataFrame:
    sheet = get_sheet_name(template_path, preferred="财务档案")
    df = pd.read_excel(template_path, sheet_name=sheet).copy()

    if "是否选中" not in df.columns:
        df["是否选中"] = ""
    if "预测值" not in df.columns:
        df["预测值"] = ""

    _mark_profile_option(df, ["二级分类", "三级分类", "四级分类"], sim["sex"])

    if sim["child_num"] == 0:
        _mark_profile_option(df, ["三级分类"], "无子女")
    elif sim["child_num"] == 1:
        _mark_profile_option(df, ["三级分类"], "1个")
    elif sim["child_num"] == 2:
        _mark_profile_option(df, ["三级分类"], "2个")
    else:
        _mark_profile_option(df, ["三级分类"], "3个及以上")

    if "二级分类" in df.columns:
        mask_age = df["二级分类"].astype(str) == "当前年龄"
        if mask_age.any():
            df.loc[mask_age, "预测值"] = f"{sim['age']}岁"

        mask_ret = df["二级分类"].astype(str) == "退休年龄"
        if mask_ret.any():
            df.loc[mask_ret, "预测值"] = f"{sim['退休年龄']}岁"

        mask_area = df["二级分类"].astype(str) == "购房面积"
        if mask_area.any() and sim["购房面积"] != "":
            df.loc[mask_area, "预测值"] = f"{sim['购房面积']}平米"

        mask_edu = df["二级分类"].astype(str) == "教育资金储备"
        if mask_edu.any() and sim["教育资金储备"] != "":
            df.loc[mask_edu, "预测值"] = f"{sim['教育资金储备']}万元"

    if sim["购房年份"] != "":
        _mark_profile_option(df, ["三级分类", "四级分类"], f"{sim['购房年份']}年")

    if sim["教育阶段"] != "":
        _mark_profile_option(df, ["三级分类"], sim["教育阶段"])

    summary_rows = pd.DataFrame({
        "一级分类": ["模拟摘要"] * 6,
        "二级分类": ["当前年龄", "家庭人数", "城乡属性", "孩子数量", "净值", "是否有房"],
        "三级分类": [np.nan] * 6,
        "四级分类": [np.nan] * 6,
        "是否选中": [""] * 6,
        "预测值": [
            f"{sim['age']}岁",
            str(sim["family_size"]),
            str(sim["urban"]),
            str(sim["child_num"]),
            round(float(sim["净值"]), 2),
            "是" if sim["自用房产"] > 0 else "否"
        ]
    })

    for col in df.columns:
        if col not in summary_rows.columns:
            summary_rows[col] = np.nan
    summary_rows = summary_rows[df.columns]

    df = pd.concat([df, summary_rows], ignore_index=True)
    return df


def build_profile_simple(sim: Dict) -> pd.DataFrame:
    total_asset = sum(sim.get(k, 0) for k in ASSET_ITEM_KEYS)
    total_debt = sum(sim.get(k, 0) for k in DEBT_ITEM_KEYS)
    total_income = sum(sim.get(k, 0) for k in INCOME_ITEM_KEYS)
    total_expense = sum(sim.get(k, 0) for k in EXPENSE_ITEM_KEYS)

    debt_ratio = total_debt / total_asset if total_asset > 0 else 0
    saving_rate = (total_income - total_expense) / total_income if total_income > 0 else 0

    return pd.DataFrame({
        "字段": [
            "当前年龄", "性别", "家庭人数", "城乡属性", "孩子数量", "是否有房",
            "退休年龄", "购房年份", "购房面积", "教育阶段", "教育资金储备",
            "总资产", "总负债", "净资产", "负债率", "年收入", "年支出", "结余率", "家庭分组"
        ],
        "值": [
            f"{sim['age']}岁",
            sim["sex"],
            sim["family_size"],
            sim["urban"],
            sim["child_num"],
            "是" if sim["自用房产"] > 0 else "否",
            f"{sim['退休年龄']}岁",
            "" if sim["购房年份"] == "" else f"{sim['购房年份']}年",
            "" if sim["购房面积"] == "" else f"{sim['购房面积']}平米",
            sim["教育阶段"],
            "" if sim["教育资金储备"] == "" else f"{sim['教育资金储备']}万元",
            round(total_asset, 2),
            round(total_debt, 2),
            round(sim["净值"], 2),
            round(debt_ratio, 4),
            round(total_income, 2),
            round(total_expense, 2),
            round(saving_rate, 4),
            sim["family_type"]
        ]
    })


# =========================================================
# 7. 结构化 6 个月汇总
# =========================================================
def build_structured_summary(monthly_dfs: List[pd.DataFrame], preferred_value_col="预测值") -> pd.DataFrame:
    """
    把 month1~month6 的模板表，按原层级做横向汇总
    """
    summary_frames = []
    for idx, df in enumerate(monthly_dfs, start=1):
        tmp = df.copy()
        value_col = preferred_value_col if preferred_value_col in tmp.columns else tmp.columns[-1]
        tmp = tmp.rename(columns={value_col: f"month{idx}"})
        keep_cols = [c for c in tmp.columns if c in ["AFP家财分类", "一级分类", "二级分类", "三级分类", "四级分类"]]
        keep_cols = keep_cols + [f"month{idx}"]
        tmp = tmp[keep_cols]
        summary_frames.append(tmp)

    # 逐月外连接
    summary = summary_frames[0]
    key_cols = [c for c in summary.columns if c.startswith("month") is False]

    for i in range(1, len(summary_frames)):
        summary = pd.merge(summary, summary_frames[i], on=key_cols, how="outer")

    month_cols = [c for c in summary.columns if c.startswith("month")]
    summary[month_cols] = summary[month_cols].fillna(0)

    # 增加 6个月合计 / 月均
    summary["6个月合计"] = summary[month_cols].sum(axis=1)
    summary["月均"] = summary[month_cols].mean(axis=1)

    return summary


# =========================================================
# 8. 保存一个用户
# =========================================================
def save_one_case(case_id: int, base_sim: Dict, grp_stats: pd.DataFrame):
    folder = os.path.join(OUTPUT_DIR, str(case_id))
    ensure_dir(folder)

    # 财务档案：静态，不受 6 个月影响
    profile_total_df = fill_profile_total(PROFILE_TEMPLATE, base_sim)
    profile_simple_df = build_profile_simple(base_sim)

    total_sheet = get_sheet_name(PROFILE_TEMPLATE, preferred="财务档案")

    with pd.ExcelWriter(os.path.join(folder, "财务档案_总表.xlsx"), engine="openpyxl") as writer:
        profile_total_df.to_excel(writer, sheet_name=total_sheet, index=False)

    with pd.ExcelWriter(os.path.join(folder, "财务档案_简表.xlsx"), engine="openpyxl") as writer:
        profile_simple_df.to_excel(writer, sheet_name="简表", index=False)

    # 6个月动态
    behavior = get_group_behavior(grp_stats, base_sim["family_type"])

    monthly_asset_dfs = []
    monthly_income_dfs = []

    for m in range(1, 7):
        month_dir = os.path.join(folder, f"month{m}")
        ensure_dir(month_dir)

        month_state = evolve_month_state(base_sim, behavior, m)

        asset_df = fill_asset_template(ASSET_TEMPLATE, month_state)
        income_df = fill_income_template(INCOME_TEMPLATE, month_state)

        monthly_asset_dfs.append(asset_df)
        monthly_income_dfs.append(income_df)

        asset_sheet = get_sheet_name(ASSET_TEMPLATE, preferred="报表")
        income_sheet = get_sheet_name(INCOME_TEMPLATE, preferred="报表")

        with pd.ExcelWriter(os.path.join(month_dir, "资产负债数据类别.xlsx"), engine="openpyxl") as writer:
            asset_df.to_excel(writer, sheet_name=asset_sheet, index=False)

        with pd.ExcelWriter(os.path.join(month_dir, "收入支出数据类别.xlsx"), engine="openpyxl") as writer:
            income_df.to_excel(writer, sheet_name=income_sheet, index=False)

    # 结构化汇总
    summary_dir = os.path.join(folder, "汇总")
    ensure_dir(summary_dir)

    asset_struct_summary = build_structured_summary(monthly_asset_dfs)
    income_struct_summary = build_structured_summary(monthly_income_dfs)

    with pd.ExcelWriter(os.path.join(summary_dir, "资产负债表_6个月结构汇总.xlsx"), engine="openpyxl") as writer:
        asset_struct_summary.to_excel(writer, sheet_name="结构汇总", index=False)

    with pd.ExcelWriter(os.path.join(summary_dir, "收入支出表_6个月结构汇总.xlsx"), engine="openpyxl") as writer:
        income_struct_summary.to_excel(writer, sheet_name="结构汇总", index=False)


# =========================================================
# 9. 主函数（带进度条）
# =========================================================
def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    ensure_dir(OUTPUT_DIR)

    must_exist = [
        ASSET_TEMPLATE, INCOME_TEMPLATE, PROFILE_TEMPLATE,
        CFPS_FILES[2018]["econ"], CFPS_FILES[2018]["conf"],
        CFPS_FILES[2020]["econ"], CFPS_FILES[2020]["conf"],
        CFPS_FILES[2022]["econ"], CFPS_FILES[2022]["conf"],
    ]
    for p in must_exist:
        if not os.path.exists(p):
            raise FileNotFoundError(f"文件不存在：{p}")

    pool, grp_stats = build_cfps_pool()
    print(f"样本池构建完成，共 {len(pool)} 条家庭样本。")

    summary_cols = [
        "year", "income", "expense_total", "total_asset_amt", "total_debt",
        "net_asset", "age", "family_size", "urban", "family_type"
    ]
    pool[summary_cols].to_excel(os.path.join(OUTPUT_DIR, "_cfps样本池摘要.xlsx"), index=False)
    grp_stats.to_excel(os.path.join(OUTPUT_DIR, "_cfps组内行为参数.xlsx"), index=False)

    for i in tqdm(range(1, N_OUTPUT + 1), desc="生成用户画像与6个月数据"):
        base_sim = simulate_base_household(pool)
        save_one_case(i, base_sim, grp_stats)

    print("全部生成完成。")


if __name__ == "__main__":
    main()

正在读取 2018 年 CFPS ...
正在读取 2020 年 CFPS ...
正在读取 2022 年 CFPS ...
样本池构建完成，共 3996 条家庭样本。


生成用户画像与6个月数据: 100%|█████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.31it/s]

全部生成完成。
